# Stage 2: Feature Engineering
**HRV Feature Extraction using Sliding Window**

---

### Mục tiêu
Trích xuất các đặc trưng biến thiên nhịp tim (Heart Rate Variability - HRV) từ chuỗi R-R Interval đã được tính ở Stage 1, sử dụng kỹ thuật **cửa sổ trượt** (Sliding Window) chuẩn y khoa.

### Pipeline
```
Clean CSV --> Sliding Window (5 min) --> Peak Detection --> R-R Intervals
                                                               |
                            +----------------------------------+
                            v
              +-------------------------+
              |  Time-Domain Features   |  Mean_NN, SDNN, RMSSD, pNN50, NN50, CV
              +-------------------------+
              |  Freq-Domain Features   |  LF, HF, LF/HF, LF_norm, HF_norm, Total_Power
              +-------------------------+
              |  SpO2 Features          |  SpO2_mean, SpO2_std, SpO2_min
              +-------------------------+
              |  Heart Rate             |  HR_mean
              +-------------------------+
                            |
                            v
                   features_dataset.csv (16 features)
```

### Tổng hợp đặc trưng (16 features)

| Nhóm | Số lượng | Đặc trưng |
|------|----------|----------|
| Time-Domain | 6 | Mean_NN, SDNN, RMSSD, pNN50, NN50, CV |
| Frequency-Domain | 6 | LF, HF, LF_HF_Ratio, LF_norm, HF_norm, Total_Power |
| SpO2 | 3 | SpO2_mean, SpO2_std, SpO2_min |
| Heart Rate | 1 | HR_mean |

### Tài liệu tham khảo
- [1] Task Force of ESC/NASPE, "Heart rate variability: Standards of measurement, physiological interpretation and clinical use," *European Heart Journal*, vol. 17, pp. 354-381, 1996.
- [2] Malik et al., "Heart rate variability in relation to prognosis after myocardial infarction," *European Heart Journal*, 1989.
- [3] Shaffer, F. & Ginsberg, J.P., "An Overview of Heart Rate Variability Metrics and Norms," *Frontiers in Public Health*, vol. 5, 2017.

## 1. Import Libraries & Load Clean Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, welch
from scipy import interpolate
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 4),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3
})

# ============================================================
# CẤU HÌNH
# ============================================================
CLEAN_FILENAME = 'clean_PHU_30_PHUT.csv'
SAMPLING_RATE  = 100.0   # Hz
WINDOW_SIZE    = 5       # phút (chuẩn Short-term HRV theo [1])
STEP_SIZE      = 1       # phút (Bước trượt)
LABEL          = 'Sitting'  # Nhãn trạng thái cho file này
# ============================================================

filepath = f'../data/processed/{CLEAN_FILENAME}'
df = pd.read_csv(filepath)

total_duration = (df['Time(ms)'].iloc[-1] - df['Time(ms)'].iloc[0]) / 60000
expected_windows = int((total_duration - WINDOW_SIZE) / STEP_SIZE) + 1

print(f'Đã đọc: {CLEAN_FILENAME}')
print(f'  Tổng dòng: {len(df):,}')
print(f'  Thời lượng: {total_duration:.1f} phút')
print(f'  Số cửa sổ dự kiến: ~{expected_windows} windows ({WINDOW_SIZE}min, step {STEP_SIZE}min)')

## 2. Định nghĩa hàm trích xuất đặc trưng

### 2.1 Time-Domain Features (Đặc trưng miền thời gian)

| Đặc trưng | Công thức | Ý nghĩa lâm sàng |
|-----------|-----------|--------------------|
| **Mean_NN** | $\bar{x} = \frac{1}{N}\sum_{i=1}^{N} RR_i$ | Khoảng R-R trung bình (ms) |
| **SDNN** | $\sqrt{\frac{1}{N-1}\sum_{i=1}^{N}(RR_i - \bar{x})^2}$ | Biến thiên tổng thể (Overall HRV) [1] |
| **RMSSD** | $\sqrt{\frac{1}{N-1}\sum_{i=1}^{N-1}(RR_{i+1} - RR_i)^2}$ | Hoạt động Parasympathetic (Vagal tone) [1] |
| **pNN50** | $\frac{NN50}{N-1} \times 100$ | Tỷ lệ % cặp R-R chênh > 50ms. Chỉ số Parasympathetic mạnh [1] |
| **NN50** | Đếm số cặp $|RR_{i+1} - RR_i| > 50ms$ | Số lượng cặp R-R chênh lệch lớn |
| **CV** | $\frac{SDNN}{Mean\_NN} \times 100$ | Hệ số biến thiên - chuẩn hóa SDNN giữa các đối tượng |

### 2.2 Frequency-Domain Features (Đặc trưng miền tần số)

Sử dụng phương pháp **Welch's Periodogram** để ước lượng mật độ phổ công suất (PSD) của chuỗi R-R Interval [1].

| Đặc trưng | Dải tần / Công thức | Ý nghĩa lâm sàng |
|-----------|---------------------|--------------------|
| **LF** | 0.04 - 0.15 Hz | Sympathetic + Parasympathetic |
| **HF** | 0.15 - 0.40 Hz | Parasympathetic (Respiratory Sinus Arrhythmia) |
| **LF/HF** | LF / HF | Cân bằng thần kinh tự chủ (Autonomic Balance) [3] |
| **LF_norm** | LF / (LF + HF) x 100 | LF chuẩn hóa - so sánh giữa các phiên đo [1] |
| **HF_norm** | HF / (LF + HF) x 100 | HF chuẩn hóa [1] |
| **Total_Power** | LF + HF | Tổng năng lượng phổ |

### 2.3 SpO2 Features (Đặc trưng nồng độ oxy máu)

| Đặc trưng | Ý nghĩa lâm sàng |
|-----------|--------------------|
| **SpO2_mean** | Nồng độ oxy trung bình trong cửa sổ |
| **SpO2_std** | Độ dao động oxy (phát hiện ngưng thở khi ngủ) |
| **SpO2_min** | Oxy thấp nhất (phát hiện desaturation) |

In [ ]:
def calculate_time_domain(rr_intervals):
    """Trích xuất đặc trưng miền thời gian từ chuỗi R-R Intervals.
    
    Parameters:
        rr_intervals (np.array): Mảng các khoảng R-R (đơn vị: ms)
    Returns:
        dict: Chứa Mean_NN, SDNN, RMSSD, pNN50, NN50, CV
    """
    if len(rr_intervals) < 2:
        return {k: np.nan for k in ['Mean_NN', 'SDNN', 'RMSSD', 'pNN50', 'NN50', 'CV']}
    
    mean_nn = np.mean(rr_intervals)
    sdnn = np.std(rr_intervals, ddof=1)
    
    diff_nn = np.diff(rr_intervals)
    rmssd = np.sqrt(np.mean(diff_nn**2))
    
    # pNN50 và NN50: Đếm các cặp R-R liên tiếp có chênh lệch > 50ms
    nn50 = np.sum(np.abs(diff_nn) > 50)
    pnn50 = (nn50 / len(diff_nn)) * 100
    
    # CV: Hệ số biến thiên (Coefficient of Variation)
    cv = (sdnn / mean_nn) * 100 if mean_nn > 0 else np.nan
    
    return {
        'Mean_NN': mean_nn,
        'SDNN': sdnn,
        'RMSSD': rmssd,
        'pNN50': pnn50,
        'NN50': nn50,
        'CV': cv
    }


def calculate_frequency_domain(rr_intervals, fs_interp=4.0):
    """Trích xuất đặc trưng miền tần số bằng Welch's Periodogram.
    
    Quy trình:
    1. Nội suy (Cubic Spline) chuỗi R-R thành tín hiệu liên tục
    2. Tính Power Spectral Density (PSD) bằng thuật toán Welch
    3. Tích phân PSD trong dải LF và HF để tính năng lượng phổ
    
    Parameters:
        rr_intervals (np.array): Mảng các khoảng R-R (đơn vị: ms)
        fs_interp (float): Tần số nội suy (mặc định 4 Hz)
    Returns:
        dict: Chứa LF, HF, LF_HF_Ratio, LF_norm, HF_norm, Total_Power
    """
    nan_result = {k: np.nan for k in ['LF', 'HF', 'LF_HF_Ratio', 'LF_norm', 'HF_norm', 'Total_Power']}
    if len(rr_intervals) < 10:
        return nan_result
    
    # Tạo trục thời gian tích lũy (giây)
    t = np.cumsum(rr_intervals) / 1000.0
    t -= t[0]
    
    # Nội suy cubic spline -> chuỗi R-R lấy mẫu đều
    t_interp = np.arange(0, t[-1], 1.0 / fs_interp)
    f_interp = interpolate.interp1d(t, rr_intervals, kind='cubic', fill_value='extrapolate')
    rr_interp = f_interp(t_interp)
    
    # Trừ trung bình (detrend) trước khi tính PSD
    rr_interp -= np.mean(rr_interp)
    
    # Welch's Periodogram
    f, pxx = welch(rr_interp, fs=fs_interp, nperseg=min(len(rr_interp), 256))
    
    # Tính năng lượng phổ trong từng dải tần
    lf_mask = (f >= 0.04) & (f < 0.15)
    hf_mask = (f >= 0.15) & (f < 0.40)
    
    lf = np.trapezoid(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
    hf = np.trapezoid(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan
    
    # Tỷ lệ LF/HF
    lf_hf_ratio = lf / hf if (hf and hf > 0) else np.nan
    
    # Normalized units (theo tiêu chuẩn [1])
    total_power = lf + hf if (not np.isnan(lf) and not np.isnan(hf)) else np.nan
    lf_norm = (lf / total_power) * 100 if (total_power and total_power > 0) else np.nan
    hf_norm = (hf / total_power) * 100 if (total_power and total_power > 0) else np.nan
    
    return {
        'LF': lf,
        'HF': hf,
        'LF_HF_Ratio': lf_hf_ratio,
        'LF_norm': lf_norm,
        'HF_norm': hf_norm,
        'Total_Power': total_power
    }


def calculate_spo2_features(spo2_values):
    """Trích xuất đặc trưng từ cột SpO2.
    
    Parameters:
        spo2_values (pd.Series): Chuỗi giá trị SpO2 trong cửa sổ
    Returns:
        dict: Chứa SpO2_mean, SpO2_std, SpO2_min
    """
    # Loại bỏ giá trị SpO2 = 0 (cảm biến chưa ổn định)
    spo2_clean = spo2_values[spo2_values > 0]
    
    if len(spo2_clean) < 10:
        return {'SpO2_mean': np.nan, 'SpO2_std': np.nan, 'SpO2_min': np.nan}
    
    return {
        'SpO2_mean': np.mean(spo2_clean),
        'SpO2_std': np.std(spo2_clean),
        'SpO2_min': np.min(spo2_clean)
    }

print('Đã định nghĩa 3 hàm trích xuất đặc trưng:')
print('  - calculate_time_domain()      -> 6 features (Mean_NN, SDNN, RMSSD, pNN50, NN50, CV)')
print('  - calculate_frequency_domain() -> 6 features (LF, HF, LF/HF, LF_norm, HF_norm, Total_Power)')
print('  - calculate_spo2_features()    -> 3 features (SpO2_mean, SpO2_std, SpO2_min)')
print('  + HR_mean                      -> 1 feature')
print('  = Tổng cộng: 16 features')

## 3. Sliding Window Feature Extraction

Quét toàn bộ dữ liệu bằng cửa sổ trượt **5 phút**, bước nhảy **1 phút** (theo khuyến nghị của [1] cho phân tích HRV ngắn hạn).

Trong mỗi cửa sổ, hệ thống thực hiện:
1. Phát hiện đỉnh nhịp tim (Peak Detection)
2. Tính R-R Intervals và lọc ngoại lai sinh lý (chỉ giữ 300-2000 ms) theo tiêu chí Malik [2]
3. Tính 16 đặc trưng (6 Time-domain + 6 Frequency-domain + 3 SpO2 + 1 HR)

Kết quả là một bảng dữ liệu nhỏ gọn, mỗi dòng tương ứng với một cửa sổ 5 phút, chứa đầy đủ các chỉ số y khoa.

In [ ]:
window_size_ms = WINDOW_SIZE * 60 * 1000
step_size_ms = STEP_SIZE * 60 * 1000

features_list = []
skipped_windows = 0

start_time = df['Time(ms)'].iloc[0]
end_time = df['Time(ms)'].iloc[-1]
current_start = start_time
window_count = 0

print(f'Đang quét dữ liệu...')
print(f'  Window: {WINDOW_SIZE} min | Step: {STEP_SIZE} min | Label: {LABEL}')
print()

while current_start + window_size_ms <= end_time:
    current_end = current_start + window_size_ms
    window_count += 1
    
    # Cắt cửa sổ
    window_df = df[(df['Time(ms)'] >= current_start) & (df['Time(ms)'] < current_end)]
    
    # Peak Detection trong cửa sổ
    min_distance = int(0.3 * SAMPLING_RATE)
    peaks, _ = find_peaks(window_df['IR_Filtered'], distance=min_distance, prominence=10)
    
    # R-R Intervals
    peak_times = window_df['Time(ms)'].iloc[peaks].values
    rr_intervals = np.diff(peak_times)
    
    # Lọc sinh lý theo tiêu chí Malik (300-2000 ms)
    rr_intervals = rr_intervals[(rr_intervals >= 300) & (rr_intervals <= 2000)]
    
    # Trích xuất đặc trưng
    if len(rr_intervals) > 30:
        # Time-domain (6 features)
        td_features = calculate_time_domain(rr_intervals)
        
        # Frequency-domain (6 features)
        fd_features = calculate_frequency_domain(rr_intervals)
        
        # SpO2 (3 features)
        spo2_features = calculate_spo2_features(window_df['SpO2'])
        
        # Heart Rate (1 feature)
        hr_mean = 60000 / td_features['Mean_NN'] if td_features['Mean_NN'] > 0 else np.nan
        
        # Gộp tất cả vào 1 dòng
        row = {
            'Window_Start(ms)': current_start,
            'Window_Start(min)': round((current_start - start_time) / 60000, 1),
            'N_beats': len(rr_intervals) + 1,
        }
        
        # Thêm Time-domain features
        for k, v in td_features.items():
            row[k] = round(v, 4) if not np.isnan(v) else np.nan
        
        # Thêm HR_mean
        row['HR_mean'] = round(hr_mean, 1) if not np.isnan(hr_mean) else np.nan
        
        # Thêm Frequency-domain features
        for k, v in fd_features.items():
            row[k] = round(v, 4) if not np.isnan(v) else np.nan
        
        # Thêm SpO2 features
        for k, v in spo2_features.items():
            row[k] = round(v, 2) if not np.isnan(v) else np.nan
        
        # Nhãn
        row['Label'] = LABEL
        
        features_list.append(row)
    else:
        skipped_windows += 1
    
    current_start += step_size_ms

features_df = pd.DataFrame(features_list)

print('=' * 60)
print('FEATURE EXTRACTION HOÀN TẤT')
print('=' * 60)
print(f'  Tổng cửa sổ quét: {window_count}')
print(f'  Cửa sổ hợp lệ:   {len(features_df)}')
print(f'  Cửa sổ bỏ qua:   {skipped_windows} (không đủ nhịp tim)')
print(f'  Số features:     {len(features_df.columns) - 4} (không tính metadata và Label)')
print(f'\nBảng Đặc Trưng HRV:')
features_df

## 4. Visualization (Trực quan hóa kết quả)

In [ ]:
if len(features_df) > 1:
    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    x = features_df['Window_Start(min)']

    # SDNN
    axes[0, 0].plot(x, features_df['SDNN'], 'o-', color='#e74c3c', markersize=4)
    axes[0, 0].set_title('SDNN (Biến thiên tổng thể)', fontweight='bold')
    axes[0, 0].set_ylabel('SDNN (ms)')

    # RMSSD
    axes[0, 1].plot(x, features_df['RMSSD'], 's-', color='#2980b9', markersize=4)
    axes[0, 1].set_title('RMSSD (Parasympathetic)', fontweight='bold')
    axes[0, 1].set_ylabel('RMSSD (ms)')

    # pNN50
    axes[1, 0].plot(x, features_df['pNN50'], '^-', color='#16a085', markersize=4)
    axes[1, 0].set_title('pNN50 (% cặp R-R chênh > 50ms)', fontweight='bold')
    axes[1, 0].set_ylabel('pNN50 (%)')

    # Heart Rate
    axes[1, 1].plot(x, features_df['HR_mean'], 'D-', color='#27ae60', markersize=4)
    axes[1, 1].set_title('Heart Rate trung bình', fontweight='bold')
    axes[1, 1].set_ylabel('HR (BPM)')

    # LF/HF Ratio
    axes[2, 0].plot(x, features_df['LF_HF_Ratio'], 'v-', color='#8e44ad', markersize=4)
    axes[2, 0].set_title('LF/HF Ratio (Autonomic Balance)', fontweight='bold')
    axes[2, 0].set_ylabel('LF/HF')
    axes[2, 0].set_xlabel('Thời gian (phút)')

    # SpO2
    axes[2, 1].plot(x, features_df['SpO2_mean'], 'p-', color='#e67e22', markersize=4)
    axes[2, 1].set_title('SpO2 trung bình', fontweight='bold')
    axes[2, 1].set_ylabel('SpO2 (%)')
    axes[2, 1].set_xlabel('Thời gian (phút)')

    plt.suptitle(f'HRV Features Over Time - Label: {LABEL}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Không đủ dữ liệu để vẽ biểu đồ (cần ít nhất 2 cửa sổ)')

In [ ]:
# Bảng thống kê mô tả toàn bộ 16 Features
feature_cols = ['Mean_NN', 'SDNN', 'RMSSD', 'pNN50', 'NN50', 'CV',
                'HR_mean', 'LF', 'HF', 'LF_HF_Ratio', 'LF_norm', 'HF_norm',
                'Total_Power', 'SpO2_mean', 'SpO2_std', 'SpO2_min']

print('Descriptive Statistics (16 Features):')
features_df[feature_cols].describe().round(4)

## 5. Export Features Dataset

In [ ]:
os.makedirs('../data/features', exist_ok=True)
save_path = '../data/features/features_dataset.csv'
features_df.to_csv(save_path, index=False)

print('=' * 60)
print('ĐÃ LƯU FILE')
print('=' * 60)
print(f'Output: {save_path}')
print(f'Kích thước: {len(features_df)} rows x {len(features_df.columns)} columns')
print(f'Số features cho ML: 16')
print(f'Label: {LABEL}')
print(f'\nSẵn sàng cho Stage 3: Model Training!')